<a href="https://colab.research.google.com/github/Atharv-1905/Machine-Learning/blob/practice/Spaceship_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
pip install xgboost

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [27]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_ids = test_df["PassengerId"]

In [28]:
def preprocess_data(df):
    df = df.copy()

    df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)

    df = df.drop(['PassengerId', 'Name', 'Cabin', 'Num'], axis=1)

    numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    categorical_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())

    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    df = pd.get_dummies(df, columns=categorical_cols)

    return df

train_processed = preprocess_data(train_df)
test_processed = preprocess_data(test_df)



/tmp/ipython-input-1732609445.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])
/tmp/ipython-input-1732609445.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


In [29]:
y = train_processed['Transported']
X = train_processed.drop('Transported', axis=1)

test_processed = test_processed.reindex(columns=X.columns, fill_value=0)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)



In [30]:
model = XGBClassifier()
print("Training model...")
model.fit(X_train, y_train)

Training model...


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [31]:
print("\n--- Model Evaluation ---")
val_predictions = model.predict(X_val)

acc = accuracy_score(y_val, val_predictions)
print(f"Accuracy: {acc:.4f} (The model is correct {acc*100:.2f}% of the time)")

cm = confusion_matrix(y_val, val_predictions)
print("\nConfusion Matrix:")
print(cm)

report = classification_report(y_val, val_predictions)
print("\nClassification Report:")
print(report)


--- Model Evaluation ---
Accuracy: 0.7861 (The model is correct 78.61% of the time)

Confusion Matrix:
[[665 196]
 [176 702]]

Classification Report:
              precision    recall  f1-score   support

       False       0.79      0.77      0.78       861
        True       0.78      0.80      0.79       878

    accuracy                           0.79      1739
   macro avg       0.79      0.79      0.79      1739
weighted avg       0.79      0.79      0.79      1739



In [32]:
final_predictions = model.predict(test_processed)
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': final_predictions.astype(bool)
})
submission.to_csv('submission.csv', index=False)
print("\nSubmission file saved: submission.csv")


Submission file saved: submission.csv
